## 1️⃣ Libraries & Setup

In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", palette="muted")

## 2️⃣ Data Paths & Loading Function
Using a single, robust function to load and clean raw data without inplace warnings.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

def resolve_raw_path(filename):
    base_name = Path(filename).name
    candidate = ROOT / "data" / "raw" / base_name
    if candidate.exists():
        return str(candidate)
    raise FileNotFoundError(f"Could not find {base_name} in {ROOT / 'data' / 'raw'}")

def load_stock_clean(filename):
    raw = pd.read_csv(resolve_raw_path(filename))
    first_col = raw.columns[0]

    # filter out non-date rows
    clean = raw.loc[raw[first_col].astype(str).str.match(r"^\\d{4}-\\d{2}-\\d{2}$", na=False)].copy()
    clean = clean.rename(columns={first_col: "Date"})

    # convert Date to datetime dtype & set index
    clean["Date"] = pd.to_datetime(clean["Date"], errors="coerce")
    clean = clean.set_index("Date")

    # convert all remaining columns to numeric
    for col in clean.columns:
        clean[col] = pd.to_numeric(clean[col], errors="coerce")

    # ffill & bfill to handle NaNs robustly without inplace=True warning
    clean = clean.ffill().bfill()

    return clean

print("Current working directory:", os.getcwd())
cleaned_dir = ROOT / "data" / "cleaned"
cleaned_dir.mkdir(parents=True, exist_ok=True)

## 3️⃣ Load Data

In [ ]:
reliance = load_stock_clean("RELIANCE_NS.csv")
tcs      = load_stock_clean("TCS_NS.csv")
infosys  = load_stock_clean("INFY_NS.csv")
hdfc     = load_stock_clean("HDFCBANK_NS.csv")

print("Datasets Loaded & Cleaned Successfully!")

## 4️⃣ Basic Validation & Info

In [ ]:
print("Type of Index:", type(reliance.index))
print("\n--- Null values in Reliance ---\n", reliance.isnull().sum())
print("\n--- RELIANCE HEAD ---\n", reliance.head())
print("\n--- RELIANCE STATS ---\n", reliance.describe().round(2))

## 5️⃣ Exploratory Visualizations

In [ ]:
plt.figure(figsize=(14,7))
plt.plot(reliance["Close"], label="RELIANCE")
plt.plot(tcs["Close"], label="TCS")
plt.plot(infosys["Close"], label="INFY")
plt.plot(hdfc["Close"], label="HDFCBANK")
plt.title("CLOSING PRICE TREND (2015–2025)")
plt.xlabel("Date")
plt.ylabel("Close Price")
plt.legend()
plt.show()

### Correlation Heatmap

In [ ]:
combined = pd.concat([
    reliance["Close"].rename("RELIANCE_Close"),
    tcs["Close"].rename("TCS_Close"),
    infosys["Close"].rename("INFY_Close"),
    hdfc["Close"].rename("HDFCBANK_Close")
], axis=1)

plt.figure(figsize=(8,6))
sns.heatmap(combined.corr(), annot=True, cmap="coolwarm")
plt.title("CORRELATION BETWEEN CLOSE PRICES")
plt.show()

### Save Cleaned Data

In [ ]:
reliance.to_csv(cleaned_dir / "reliance_clean.csv")
tcs.to_csv(cleaned_dir / "tcs_clean.csv")
infosys.to_csv(cleaned_dir / "infosys_clean.csv")
hdfc.to_csv(cleaned_dir / "hdfcbank_clean.csv")

print("Cleaned data saved to data/cleaned/")